# DistilBERT — human-written deceptive-review detector (Colab GPU)

Fine-tunes `distilbert-base-uncased` on the **Ott + Li 2014** human-deception corpus to detect **human-written** fake reviews (distinct from Model A's AI/CG detector and the Yelp behavioral/GNN models). The DL counterpart to `train_human_fake_sklearn.py` (sklearn best = LR 88.2% macro-F1).

Small dataset (~2.3k train) → few epochs, **early stopping on val macro-F1** (restore best) + weighted cross-entropy for the mild class imbalance.

**Setup:** GPU runtime (Runtime → Change runtime type → GPU). Run cell 1 and **upload `data/human_fake/human_fake_reviews.csv`** when prompted. Run all.

In [ ]:
from google.colab import files
up = files.upload()   # choose human_fake_reviews.csv
CSV = list(up.keys())[0]
print('uploaded:', CSV)

In [ ]:
import os
os.environ.setdefault('USE_TF', '0'); os.environ.setdefault('USE_TORCH', '1')   # torch backend only
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, recall_score, precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME, MAX_LEN, BATCH, LR, MAX_EPOCHS, PATIENCE = 'distilbert-base-uncased', 256, 16, 2e-5, 8, 2
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:', dev)

df = pd.read_csv(CSV); df['text'] = df['text'].astype(str)
tr, va, te = (df[df.split==s].reset_index(drop=True) for s in ('train','val','test'))
print(f'train {len(tr):,} / val {len(va):,} / test {len(te):,}  '
      f'(test: {int((te.label==1).sum())} fake / {int((te.label==0).sum())} genuine)')

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(dev)
def enc(texts):
    e = tok(list(texts), truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
    return e['input_ids'], e['attention_mask']
def loader(d, shuffle):
    ids, am = enc(d.text); y = torch.tensor(d.label.values)
    return DataLoader(TensorDataset(ids, am, y), batch_size=BATCH, shuffle=shuffle)
tr_dl, va_dl, te_dl = loader(tr, True), loader(va, False), loader(te, False)
n0, n1 = (tr.label==0).sum(), (tr.label==1).sum()
w = torch.tensor([len(tr)/(2*n0), len(tr)/(2*n1)], dtype=torch.float, device=dev)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

@torch.no_grad()
def evaluate(dl):
    model.eval(); yt, yp = [], []
    for ids, am, y in dl:
        logit = model(input_ids=ids.to(dev), attention_mask=am.to(dev)).logits
        yp += logit.argmax(1).cpu().tolist(); yt += y.tolist()
    return (f1_score(yt,yp,average='macro',zero_division=0),
            recall_score(yt,yp,pos_label=1,zero_division=0),
            precision_score(yt,yp,pos_label=1,zero_division=0))

best, best_state, bad = -1, None, 0
for ep in range(1, MAX_EPOCHS+1):
    model.train()
    for ids, am, y in tr_dl:
        opt.zero_grad()
        logit = model(input_ids=ids.to(dev), attention_mask=am.to(dev)).logits
        F.cross_entropy(logit, y.to(dev), weight=w).backward(); opt.step()
    vmf1 = evaluate(va_dl)[0]; print(f'epoch {ep}: val macro-F1 {vmf1:.1%}')
    if vmf1 > best: best, best_state, bad = vmf1, {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}, 0
    else: bad += 1
    if bad >= PATIENCE: print(f'early stop at epoch {ep}'); break
model.load_state_dict(best_state)

In [ ]:
mf1, fr, fp = evaluate(te_dl)
print('='*56)
print(f'DistilBERT human-deception — TEST: macro-F1 {mf1:.1%} | fake recall {fr:.1%} | fake precision {fp:.1%}')
print('='*56)
print('(sklearn reference on the same split: LR 88.2%, ensemble 86.6%)')

# Optional: save + download the fine-tuned model to drop into data/human_fake/human_fake_distilbert/
model.save_pretrained('human_fake_distilbert'); tok.save_pretrained('human_fake_distilbert')
import shutil; shutil.make_archive('human_fake_distilbert', 'zip', 'human_fake_distilbert')
from google.colab import files; files.download('human_fake_distilbert.zip')